# Simple VLA hands-on

`PLAN.md` の流れに沿って，各部のアーキテクチャを個別ファイルから読み込みます。標準デモは軽量な合成 pick-and-place データで動きます。Genesis や Qwen を接続するときも，Dataset と条件トークン生成器を差し替えるだけにしています。

In [ ]:
try:
    import google.colab
    ip = get_ipython()
    ip.run_line_magic("cd", "/content")
    ip.system("rm -rf simple-vla")
    ip.system("git clone https://github.com/Azuma413/simple-vla.git") # 自分のリポジトリに置き換えてください
    ip.run_line_magic("cd", "/content/simple-vla")
    ip.system("curl -LsSf https://astral.sh/uv/install.sh | sh")
    ip.system("uv pip install --system -e .")
except ImportError:
    print("Not running on Colab; skipping Colab setup.")

In [ ]:
import torch
from torch.utils.data import DataLoader

from part1_vision import ColoredSquareDataset, SmallVisionEncoder, train_vision_epoch, evaluate_vision
from part2_simulator import TinyPickPlaceConfig, TinyPickPlaceDataset, collate_tiny_pick_place
from part3_bc import StateMLPPolicy, ImageBCPolicy, train_bc_epoch, evaluate_bc_mse
from part4_transformer_dit import ChunkTransformerPolicy, train_chunk_epoch, evaluate_chunk_mse
from part5_flow_dit import FlowMatchingDiT, train_flow_epoch
from part6_vla_connector import VLAConnectorPolicy, train_vla_epoch

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
device

## 第1部: 視覚エンコーダ

`part1_vision.py` は，クラス分類・座標回帰・後段用パッチトークンを返す小さな CNN です。

In [ ]:
vision_train = DataLoader(ColoredSquareDataset(n=256, seed=0), batch_size=32, shuffle=True)
vision_test = DataLoader(ColoredSquareDataset(n=128, seed=1), batch_size=64)

vision = SmallVisionEncoder().to(device)
opt = torch.optim.Adam(vision.parameters(), lr=1e-3)
for epoch in range(3):
    loss = train_vision_epoch(vision, vision_train, opt, device)
    print(epoch, loss, evaluate_vision(vision, vision_test, device))

## 第2部: 教師データ

`part2_simulator.py` は Genesis の代わりに，画像・低次元状態・言語指示・アクションチャンクを持つ最小データセットを作ります。

In [ ]:
config = TinyPickPlaceConfig(n_episodes=512, seed=2)
train_data = TinyPickPlaceDataset(config=config, chunk_size=4)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_tiny_pick_place)
batch = next(iter(train_loader))
{k: (v.shape if isinstance(v, torch.Tensor) else v[:2]) for k, v in batch.items()}

## 第3部: 1-step BC

`part3_bc.py` は，低次元状態 MLP と画像 CNN+MLP の素朴なベースラインです。

In [ ]:
state_policy = StateMLPPolicy().to(device)
state_opt = torch.optim.Adam(state_policy.parameters(), lr=1e-3)
for epoch in range(3):
    loss = train_bc_epoch(state_policy, train_loader, state_opt, input_key="state", device=device)
    print("state", epoch, loss, evaluate_bc_mse(state_policy, train_loader, input_key="state", device=device))

image_policy = ImageBCPolicy().to(device)
image_opt = torch.optim.Adam(image_policy.parameters(), lr=1e-3)
for epoch in range(3):
    loss = train_bc_epoch(image_policy, train_loader, image_opt, input_key="image", device=device)
    print("image", epoch, loss, evaluate_bc_mse(image_policy, train_loader, input_key="image", device=device))

## 第4部: 単一ストリーム Transformer

`part4_transformer_dit.py` は `[CNN パッチトークン, アクショントークン]` を連結し，ブロックマスク付き self-attention でアクションチャンクを出します。

In [ ]:
chunk_policy = ChunkTransformerPolicy(chunk_size=4).to(device)
chunk_opt = torch.optim.Adam(chunk_policy.parameters(), lr=1e-3)
for epoch in range(3):
    loss = train_chunk_epoch(chunk_policy, train_loader, chunk_opt, device=device)
    print(epoch, loss, evaluate_chunk_mse(chunk_policy, train_loader, device=device))

## 第5部: Flow Matching DiT

`part5_flow_dit.py` は，clean action と Gaussian noise の補間点から速度場を回帰します。推論はノイズから Euler 積分します。

In [ ]:
flow = FlowMatchingDiT(chunk_size=4).to(device)
flow_opt = torch.optim.Adam(flow.parameters(), lr=1e-3)
for epoch in range(3):
    loss = train_flow_epoch(flow, train_loader, flow_opt, device=device)
    print(epoch, loss)

with torch.no_grad():
    sample = flow.sample(batch["image"][:4].to(device), steps=8)
sample.shape, sample[0]

## 第6部: VLA 接続

`part6_vla_connector.py` は，凍結 VLM 風の hidden state を線形層で DiT 次元へ射影します。実 Qwen にする場合も，`FrozenTinyVLM.forward()` と同じ形の hidden 列を返せば DiT 側は変えません。

In [ ]:
vla = VLAConnectorPolicy(chunk_size=4).to(device)
vla_opt = torch.optim.Adam([p for p in vla.parameters() if p.requires_grad], lr=1e-3)
for epoch in range(3):
    loss = train_vla_epoch(vla, train_loader, vla_opt, device=device)
    print(epoch, loss)

with torch.no_grad():
    pred = vla(batch["image"][:4].to(device), batch["instruction"][:4])
pred.shape, pred[0]